# T4 memory peak diagnostic

Run on Kaggle with a Tesla T4 and Internet enabled. The default run creates the historical dense diagnostic file, with every public training row truncated at 512 tokens without padding, then runs `harness/run_task.py --only solution` through the external memory recorder. Use `MODE = "public"` to profile the public training file unchanged.

The run writes the harness log, per-step table, and raw PyTorch snapshot under `/kaggle/working/t4_mempeak_<mode>/`.

In [ ]:
REPO_URL = "https://github.com/4spanch1k/soup-kernel-challenge.git"
BRANCH = "c17/mempeak"
MODE = "dense"  # dense or public

In [ ]:
import subprocess

gpus = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True, check=True,
).stdout
print(gpus)
assert "T4" in gpus, "Select a Kaggle Tesla T4 accelerator"
!python -m pip install -q "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" "bitsandbytes==0.50.2" "accelerate==1.14.0"

In [ ]:
import shutil
from pathlib import Path

repo_dir = Path("/kaggle/working/repo")
shutil.rmtree(repo_dir, ignore_errors=True)
clone = subprocess.run(
    ["git", "clone", "--quiet", "--branch", BRANCH, "--single-branch", REPO_URL, str(repo_dir)],
    capture_output=True, text=True,
)
if clone.returncode:
    raise SystemExit("git clone failed: " + clone.stderr)
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--quiet", BRANCH], check=True)
print(subprocess.run(
    ["git", "-C", str(repo_dir), "log", "-1", "--format=%H %s"],
    capture_output=True, text=True, check=True,
).stdout.strip())

In [ ]:
if MODE not in {"dense", "public"}:
    raise ValueError("MODE must be 'dense' or 'public'")
subprocess.run(["bash", "scripts/t4_mem2.sh", MODE], cwd=repo_dir, check=True)